# Gaia + NASA Data Fetching Workflow

This notebook prepares the raw data for the exoplanet-host prediction project.

Main steps:
1. Download confirmed exoplanet-host stellar data from NASA Exoplanet Archive.
2. Extract Gaia DR3 source IDs from the NASA table.
3. Fetch Gaia DR3 stellar parameters for known host IDs.
4. Fetch Gaia DR3 negative samples for training.
5. Build the final labeled training dataset.
6. Optionally fetch unseen Gaia stars for future prediction/top-candidate search.



In [1]:
from pathlib import Path
import sys
import pandas as pd

# If this notebook is inside /notebooks, make imports from project root work.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.fetch_data import (
    download_nasa_stellar,
    extract_gaia_ids,
    fetch_gaia,
    build_labeled_training_dataset,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).


(PosixPath('/Users/semyonsidorov/exoplanets_project/data/raw'),
 PosixPath('/Users/semyonsidorov/exoplanets_project/data/processed'))

## Gaia login

For small public queries, Gaia may work without login, but login is useful for larger jobs.


In [2]:
from astroquery.gaia import Gaia

# Uncomment when needed. 
# Gaia.login(user="", password="")


INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


## 1. Download NASA confirmed host-star data

NASA `pscomppars` gives confirmed exoplanet systems and stellar parameters. This is the positive class source.


In [3]:
nasa_df = download_nasa_stellar(
    output_path=RAW_DIR / "Stellar_Host.csv"
)

nasa_df.head()

Saved to: /Users/semyonsidorov/exoplanets_project/data/raw/Stellar_Host.csv


,gaia_dr3_id,st_teff,st_rad,st_mass,st_met,st_logg,st_lum,st_age,sy_dist,sy_plx
0,Gaia DR3 2086441721667415424,4971.0,0.750,0.790,-0.05,4.600,-0.53589,4.27,820.905,1.190750
1,Gaia DR3 2052381737659167488,5705.0,0.905,0.943,-0.06,4.499,-0.07942,NaN,1061.770,0.914400
2,Gaia DR3 2100418850915010432,6022.0,1.230,1.120,0.07,4.310,0.39085,4.17,493.175,1.999070
3,Gaia DR3 2078049767888231936,6747.0,1.810,1.490,0.08,4.090,0.71041,1.62,1318.050,0.730368
4,Gaia DR3 2136207095405014912,5446.0,0.821,0.824,-0.20,4.525,-0.39819,7.20,962.888,1.010300


In [4]:
nasa_df.info()
nasa_df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 6291 entries, 0 to 6290
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   gaia_dr3_id  5913 non-null   str    
 1   st_teff      5997 non-null   float64
 2   st_rad       5973 non-null   float64
 3   st_mass      6282 non-null   float64
 4   st_met       5652 non-null   float64
 5   st_logg      5969 non-null   float64
 6   st_lum       5979 non-null   float64
 7   st_age       4866 non-null   float64
 8   sy_dist      6264 non-null   float64
 9   sy_plx       5897 non-null   float64
dtypes: float64(9), str(1)
memory usage: 491.6 KB


gaia_dr3_id     378
st_teff         294
st_rad          318
st_mass           9
st_met          639
st_logg         322
st_lum          312
st_age         1425
sy_dist          27
sy_plx          394
dtype: int64

## 2. Extract Gaia DR3 IDs from NASA data

These IDs allow us to query Gaia DR3 for additional/fallback stellar parameters for known exoplanet-host stars.


In [5]:
gaia_ids = extract_gaia_ids(nasa_df)

print(f"Extracted Gaia IDs: {len(gaia_ids)}")
print(gaia_ids[:5])

Extracted Gaia IDs: 4359
[2086441721667415424, 2052381737659167488, 2100418850915010432, 2078049767888231936, 2136207095405014912]


## 3. Fetch Gaia data for known host stars

`mode="ids"` queries Gaia only for the known Gaia source IDs extracted from NASA.



In [6]:
gaia_hosts_df = fetch_gaia(
    mode="ids",
    gaia_ids=gaia_ids,
    batch_size=1500,
    output_path=RAW_DIR / "Gaia_Hosts.csv",
)

gaia_hosts_df.head()

Batch 1/3 done — 960 stars
Batch 2/3 done — 996 stars
Batch 3/3 done — 856 stars

Fetched: 2812 unique stars
Null counts:
gaia_dr3_id    0
st_teff        0
st_logg        0
st_met         0
sy_dist        0
sy_plx         0
st_rad         0
st_mass        0
st_lum         0
st_age         0
dtype: int64

Saved to: /Users/semyonsidorov/exoplanets_project/data/raw/Gaia_Hosts.csv


,gaia_dr3_id,st_teff,st_logg,st_met,sy_dist,sy_plx,st_rad,st_mass,st_lum,st_age
0,525352325110310400,4933.496582,4.5254,0.4377,176.518402,5.664558,0.8505,0.808742,-0.409362,10.982633
1,581100382135253760,5556.243164,4.4210,-0.2192,30.291500,32.977697,0.9861,0.907295,-0.082056,10.149987
2,2052735368085096576,5700.598145,4.3902,-0.1397,1386.824951,0.619297,0.9791,0.939247,0.048097,10.816549
3,2052754540819149568,5035.206543,4.5667,0.1493,904.571777,0.981293,0.7880,0.803236,-0.356144,12.040087
4,2053579930453436800,5465.395020,4.3898,-0.6747,1332.398926,0.692106,0.9998,0.880275,-0.055679,12.365674


## 4. Fetch Gaia negative training sample

`mode="negatives"` fetches Gaia stars with complete parameters and adds:

```python
is_exoplanet_host = 0
```


In [7]:
cool_m_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=2500,
    temp_max=3300,
    output_path=RAW_DIR/"gaia_negatives_cool_m.csv"
)

m_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=3300,
    temp_max=3900,
    output_path=RAW_DIR/"gaia_negatives_m.csv"
)

k_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=3900,
    temp_max=5300,
    output_path=RAW_DIR/"gaia_negatives_k.csv"
)

g_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=5300,
    temp_max=6000,
    output_path=RAW_DIR/"gaia_negatives_g.csv"
)

f_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=6000,
    temp_max=7300,
    output_path=RAW_DIR/"gaia_negatives_f.csv"
)

a_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=7300,
    temp_max=10_000,
    output_path=RAW_DIR/"gaia_negatives_a.csv"
)

b_negatives = fetch_gaia(
    mode="negatives",
    n_stars=3000,
    temp_min=10000,
    temp_max=33000,
    output_path=RAW_DIR/"gaia_negatives_b.csv"
)

balanced_negatives = pd.concat(
    [
        cool_m_negatives,
        m_negatives,
        k_negatives,
        g_negatives,
        f_negatives,
        b_negatives,
        a_negatives,
    ],
    ignore_index=True
).drop_duplicates(subset="gaia_dr3_id")

balanced_negatives["is_exoplanet_host"] = 0

balanced_negatives.to_csv(
    PROCESSED_DIR/"gaia_negatives_balanced_by_temperature.csv",
    index=False
)



Fetching 3000 Gaia stars for mode='negatives'...

Fetched: 3000 unique stars
Null counts:
gaia_dr3_id          0
st_teff              0
st_logg              0
st_met               0
sy_dist              0
sy_plx               0
st_rad               0
st_mass              0
st_lum               0
st_age               0
is_exoplanet_host    0
dtype: int64

Saved to: /Users/semyonsidorov/exoplanets_project/data/raw/gaia_negatives_cool_m.csv
Fetching 3000 Gaia stars for mode='negatives'...

Fetched: 3000 unique stars
Null counts:
gaia_dr3_id          0
st_teff              0
st_logg              0
st_met               0
sy_dist              0
sy_plx               0
st_rad               0
st_mass              0
st_lum               0
st_age               0
is_exoplanet_host    0
dtype: int64

Saved to: /Users/semyonsidorov/exoplanets_project/data/raw/gaia_negatives_m.csv
Fetching 3000 Gaia stars for mode='negatives'...

Fetched: 3000 unique stars
Null counts:
gaia_dr3_id          0
st_teff 

## 5. Build final labeled training dataset




In [8]:
# balanced_negatives = balanced_negatives[
#     (balanced_negatives["st_rad"] < 5) &
#     (balanced_negatives["st_met"] > -0.5)
# ].copy()

labeled_stars_df = build_labeled_training_dataset(
    nasa_df=nasa_df,
    gaia_hosts_df=gaia_hosts_df,
    gaia_negatives_df=balanced_negatives,
    output_path=PROCESSED_DIR / "Labeled_Stars.csv",
)

balanced_negatives.shape

Total stars: 24864
Hosts: 3864
Non-hosts: 21000
Saved to: /Users/semyonsidorov/exoplanets_project/data/processed/Labeled_Stars.csv


(21000, 11)

In [9]:
labeled_stars_df.info()
labeled_stars_df["is_exoplanet_host"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 24864 entries, 0 to 24863
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gaia_dr3_id        24864 non-null  int64  
 1   st_teff            24864 non-null  float64
 2   st_rad             24864 non-null  float64
 3   st_mass            24864 non-null  float64
 4   st_met             24864 non-null  float64
 5   st_logg            24864 non-null  float64
 6   st_lum             24864 non-null  float64
 7   st_age             24864 non-null  float64
 8   sy_dist            24864 non-null  float64
 9   sy_plx             24864 non-null  float64
 10  is_exoplanet_host  24864 non-null  int64  
dtypes: float64(9), int64(2)
memory usage: 2.1 MB


is_exoplanet_host
0    21000
1     3864
Name: count, dtype: int64